<a href="https://colab.research.google.com/github/aeau/MAU-AML-labs/blob/develop/2-language-models-lab/1-word2vec.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Word2Vec ## 

In this notebook we will go through the step by step creation of the Continouous Bag Of Words (CBOW).
CBOW is an embedded model that makes use of a "fake task" -> [within short window, predict the current word] to extract a vector that shows the relationship between words.

### Continuous Bag Of Words ###

Adapted from Robert Guthrie

In [2]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.linalg

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = "cpu"

In [4]:
# CBOW is a window view; we are trying to infer the word in the middle.
CONTEXT_SIZE = 2  # 2 words to the left, 2 to the right

raw_text= """Long Short-Term Memory (LSTM) is a recurrent neural network (RNN) architecture that has been designed
to address the vanishing and exploding gradient problems of conventional RNNs. Unlike feedforward neural networks,
RNNs have cyclic connections making them powerful for modeling sequences. 
They have been successfully used for sequence labeling and sequence prediction tasks,
such as handwriting recognition, language modeling, phonetic labeling of acoustic frames. However, in contrast to the deep neural
networks, the use of RNNs in speech recognition has been limited to phone recognition in small scale tasks. 
In this paper, we present novel LSTM based RNN architectures which make more effective
use of model parameters to train acoustic models for large vocabulary speech recognition. 
We train and compare LSTM, RNN and DNN models at various numbers of parameters and configurations.
We show that LSTM models converge quickly and give state of the art speech recognition performance for relatively small sized models.""".split()

# By deriving a set from "raw_text", we deduplicate the array
vocab = set(raw_text)
vocab_size = len(vocab)

# Basic Tokenizer
word_to_ix = {word: i for i, word in enumerate(vocab)}

print(len(raw_text))
print(vocab_size)

152
106


In [5]:
vocab

{'(LSTM)',
 '(RNN)',
 'DNN',
 'However,',
 'In',
 'LSTM',
 'LSTM,',
 'Long',
 'Memory',
 'RNN',
 'RNNs',
 'RNNs.',
 'Short-Term',
 'They',
 'Unlike',
 'We',
 'a',
 'acoustic',
 'address',
 'and',
 'architecture',
 'architectures',
 'art',
 'as',
 'at',
 'based',
 'been',
 'compare',
 'configurations.',
 'connections',
 'contrast',
 'conventional',
 'converge',
 'cyclic',
 'deep',
 'designed',
 'effective',
 'exploding',
 'feedforward',
 'for',
 'frames.',
 'give',
 'gradient',
 'handwriting',
 'has',
 'have',
 'in',
 'is',
 'labeling',
 'language',
 'large',
 'limited',
 'make',
 'making',
 'model',
 'modeling',
 'modeling,',
 'models',
 'models.',
 'more',
 'network',
 'networks,',
 'neural',
 'novel',
 'numbers',
 'of',
 'paper,',
 'parameters',
 'performance',
 'phone',
 'phonetic',
 'powerful',
 'prediction',
 'present',
 'problems',
 'quickly',
 'recognition',
 'recognition,',
 'recognition.',
 'recurrent',
 'relatively',
 'scale',
 'sequence',
 'sequences.',
 'show',
 'sized',
 '

In [6]:
word_to_ix

{'tasks,': 0,
 'make': 1,
 'have': 2,
 'to': 3,
 'tasks.': 4,
 'model': 5,
 'architecture': 6,
 'train': 7,
 'of': 8,
 'deep': 9,
 'scale': 10,
 'in': 11,
 'address': 12,
 'network': 13,
 'acoustic': 14,
 'connections': 15,
 'sequences.': 16,
 'parameters': 17,
 'large': 18,
 'phone': 19,
 'use': 20,
 'give': 21,
 'has': 22,
 'modeling': 23,
 'feedforward': 24,
 'vanishing': 25,
 'Memory': 26,
 'powerful': 27,
 'sequence': 28,
 'designed': 29,
 'a': 30,
 'RNNs': 31,
 'However,': 32,
 'art': 33,
 'them': 34,
 'conventional': 35,
 'prediction': 36,
 'contrast': 37,
 'novel': 38,
 'making': 39,
 'recognition,': 40,
 'been': 41,
 'They': 42,
 'LSTM,': 43,
 'small': 44,
 'for': 45,
 '(LSTM)': 46,
 'based': 47,
 'In': 48,
 'numbers': 49,
 'state': 50,
 'we': 51,
 'more': 52,
 'which': 53,
 'models.': 54,
 'is': 55,
 'various': 56,
 'effective': 57,
 'performance': 58,
 'successfully': 59,
 'problems': 60,
 'used': 61,
 'as': 62,
 'such': 63,
 'We': 64,
 'paper,': 65,
 'models': 66,
 'handwri

In [7]:
print(vocab)

{'tasks,', 'make', 'have', 'to', 'tasks.', 'model', 'architecture', 'train', 'of', 'deep', 'scale', 'in', 'address', 'network', 'acoustic', 'connections', 'sequences.', 'parameters', 'large', 'phone', 'use', 'give', 'has', 'modeling', 'feedforward', 'vanishing', 'Memory', 'powerful', 'sequence', 'designed', 'a', 'RNNs', 'However,', 'art', 'them', 'conventional', 'prediction', 'contrast', 'novel', 'making', 'recognition,', 'been', 'They', 'LSTM,', 'small', 'for', '(LSTM)', 'based', 'In', 'numbers', 'state', 'we', 'more', 'which', 'models.', 'is', 'various', 'effective', 'performance', 'successfully', 'problems', 'used', 'as', 'such', 'We', 'paper,', 'models', 'handwriting', 'this', 'present', 'speech', 'sized', 'Unlike', 'exploding', 'configurations.', 'and', 'frames.', 'quickly', '(RNN)', 'compare', 'show', 'that', 'RNN', 'labeling', 'the', 'LSTM', 'RNNs.', 'at', 'modeling,', 'neural', 'Long', 'networks,', 'recurrent', 'phonetic', 'converge', 'relatively', 'cyclic', 'recognition.', 'Sh

In [8]:
# list out keys and values separately
key_list = list(word_to_ix.keys())
val_list = list(word_to_ix.values())

In [ ]:
# create a "dataset"
data = []
for i in range(CONTEXT_SIZE, len(raw_text) - CONTEXT_SIZE):
    context = []
    for j in range(CONTEXT_SIZE, 0, -1):
        context.append(raw_text[i - j])

    for j in range(1, CONTEXT_SIZE + 1):
        context.append(raw_text[i + j])
        
    target = raw_text[i]
    data.append((context, target))
print(data[:5])


[(['Long', 'Short-Term', '(LSTM)', 'is'], 'Memory'), (['Short-Term', 'Memory', 'is', 'a'], '(LSTM)'), (['Memory', '(LSTM)', 'a', 'recurrent'], 'is'), (['(LSTM)', 'is', 'recurrent', 'neural'], 'a'), (['is', 'a', 'neural', 'network'], 'recurrent')]


### Create the CBOW Model (as we have seen already other ANN) ###

We have to extend from nn.Module as all the other ANN

In [10]:
class CBOW(nn.Module):

    def __init__(self, vocab_size, embed_dim, context, hidden_size):
        super(CBOW, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.linear = nn.Sequential(
            nn.Linear(context*embed_dim, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, vocab_size),
            nn.LogSoftmax(dim = -1)
        )
        
    def forward(self, inputs):
#         print(inputs.shape)
#         print(inputs)
        out = self.embedding(inputs)
#         print(out.shape)
        out = out.view(1, -1)
#         print(out.shape)
        out = self.linear(out)
#         print(out.shape)
        return out
    
    # This is what we are actually interested in
    def get_word_vector(self, word):
        out = self.embedding(word)
        return out


#### Lets break it down! ####

In [11]:
VOCAB_SIZE = len(vocab)
EMBEDD_DIM = 10
BATCH_SIZE = 6
FULL_CONTEXT_SIZE = CONTEXT_SIZE * 2
HIDDEN_SIZE = 256

example_tensor = torch.randint(0, VOCAB_SIZE, [BATCH_SIZE, FULL_CONTEXT_SIZE])
print(example_tensor)

tensor([[ 32,  58,  53,  40],
        [ 62,  26, 105,  61],
        [101,  43,  91,  91],
        [  2,  23,  70,  21],
        [ 18,  23,   8,  32],
        [ 57,  61,  23,  54]])


In [12]:
CBOW_embedding = nn.Embedding(VOCAB_SIZE, EMBEDD_DIM)
print(example_tensor.shape)
example_result = CBOW_embedding(example_tensor)
# Now we have a representation of the words in a vector of EMBEDD_DIM Dimensions
print(example_result.shape)
# example_result = torch.flatten(example_result, start_dim=1)
example_result = example_result.view(BATCH_SIZE, -1)
print(example_result.shape)

torch.Size([6, 4])
torch.Size([6, 4, 10])
torch.Size([6, 40])


In [13]:
print("input shape: ", EMBEDD_DIM * FULL_CONTEXT_SIZE)
print("output shape: ", HIDDEN_SIZE)
CBOW_hidden = nn.Linear(EMBEDD_DIM * FULL_CONTEXT_SIZE, HIDDEN_SIZE)
CBOW_hidden_relu = nn.ReLU()
example_result = CBOW_hidden(example_result)
example_result = CBOW_hidden_relu(example_result)
print(example_result.shape)

input shape:  40
output shape:  256
torch.Size([6, 256])


In [14]:
CBOW_output = nn.Linear(HIDDEN_SIZE, VOCAB_SIZE)
CBOW_output_soft = nn.LogSoftmax(dim = -1)
example_result = CBOW_output(example_result)
example_result = CBOW_output_soft(example_result)
print(example_result.shape)

torch.Size([6, 106])


In [15]:
print(example_result[0].argmax(-1))
print(key_list[val_list.index(example_result[0].argmax(-1))])
print(example_result[0])
print(example_result[1].argmax(-1))
print(key_list[val_list.index(example_result[1].argmax(-1))])
print(example_result[1])
# print(example_result[2].argmax(-1))
# print(example_result[3].argmax(-1))
# print(example_result[4].argmax(-1))

tensor(105)
recognition
tensor([-4.9655, -5.0030, -4.7502, -5.0511, -5.1420, -4.5417, -4.8386, -4.7371,
        -4.5473, -4.6109, -4.8967, -4.4097, -5.0412, -4.8166, -4.7015, -4.8762,
        -4.6955, -4.9490, -4.8514, -4.7288, -4.3130, -4.6452, -4.3547, -4.9257,
        -4.5287, -5.0509, -4.8348, -5.1938, -4.2373, -4.9473, -4.4854, -4.4864,
        -5.0278, -4.6062, -4.7552, -4.2865, -4.9708, -4.3284, -4.9939, -4.9195,
        -4.6815, -4.6433, -4.3990, -4.9345, -4.8080, -5.0959, -4.5359, -4.6658,
        -5.2985, -4.3547, -4.6091, -4.8042, -4.4092, -4.3727, -4.7977, -4.9343,
        -4.5175, -5.3308, -4.7575, -4.4824, -4.7085, -4.6603, -5.0354, -5.1869,
        -4.7600, -4.7398, -4.5897, -4.9532, -4.6729, -4.7487, -4.4472, -4.4193,
        -4.6212, -5.1386, -4.7705, -4.4499, -4.3459, -4.6826, -4.8279, -4.5659,
        -4.4885, -4.9071, -4.0581, -4.4302, -4.6201, -4.6789, -4.7768, -4.7876,
        -4.5107, -5.1772, -4.6025, -4.7668, -4.9515, -4.6334, -4.2193, -4.2180,
        -5.3056,

## Back to the notebook ##

In [16]:
# Simple helper method to transform the context to the expected int vector - tensor

def make_context_vector(context, word_to_ix, debug=False):
    if debug:
      print(context)
    idxs = [word_to_ix[w] for w in context]
    return torch.tensor(idxs, dtype=torch.long)

make_context_vector(data[0][0], word_to_ix, debug=True)

['Long', 'Short-Term', '(LSTM)', 'is']


tensor([90, 98, 46, 55])

In [17]:
def train(model, epochs, data, optimizer, loss_fn):
    model.train()
    losses = []
    for epoch in range(epochs):
        total_loss = 0
        for context, target in data:

            # Prepare inputs and targets 
            context_idxs = make_context_vector(context, word_to_ix)
            context_idxs = context_idxs.to(device)
            target_id = make_context_vector([target], word_to_ix)
            target_id = target_id.to(device)

            # Do not accumulate 
            model.zero_grad()

            # Step 3. Run the forward pass
            log_probs = model(context_idxs)
    #         break

            # Step 4. Compute your loss function.
            loss = loss_fn(log_probs, target_id)

    #         loss = loss_function(log_probs, torch.tensor([word_to_ix[target]], dtype=torch.long))

            # Step 5. Do the backward pass and update the gradient
            loss.backward()
            optimizer.step()

            # Get the Python number from a 1-element Tensor by calling tensor.item()
            total_loss += loss.item()
        losses.append(total_loss)
    return losses
    

In [18]:
VOCAB_SIZE = len(vocab)
EMBEDD_DIM = 10
BATCH_SIZE = 6
FULL_CONTEXT_SIZE = CONTEXT_SIZE * 2
HIDDEN_SIZE = 256

loss_function = nn.NLLLoss() # Because we are using Log_softmax
model = CBOW(vocab_size, EMBEDD_DIM, FULL_CONTEXT_SIZE, HIDDEN_SIZE)
model = model.to(device)
optimizer = optim.SGD(model.parameters(), lr=0.001)

losses = train(model, 100, data, optimizer, loss_function)
model.eval()

print(losses)  # The loss decreased every iteration over the training data!

[693.7475361824036, 687.113118648529, 680.5696840286255, 674.1133363246918, 667.7379422187805, 661.4366610050201, 655.2050364017487, 649.04079413414, 642.9392561912537, 636.8951029777527, 630.9040369987488, 624.9621942043304, 619.0674397945404, 613.217253446579, 607.4054048061371, 601.6306457519531, 595.8897285461426, 590.1785199642181, 584.4930546283722, 578.8298976421356, 573.1817963123322, 567.5486969947815, 561.9275929927826, 556.3143975734711, 550.7082259654999, 545.1084432601929, 539.5066339969635, 533.9043372869492, 528.301315665245, 522.6932187080383, 517.077672123909, 511.4530062675476, 505.8170232772827, 500.17029798030853, 494.5097938776016, 488.83799505233765, 483.15285336971283, 477.4564918279648, 471.7432345151901, 466.01673716306686, 460.27820068597794, 454.52279567718506, 448.75503784418106, 442.9722902774811, 437.17234325408936, 431.35785472393036, 425.53079557418823, 419.6896344423294, 413.83672416210175, 407.97529196739197, 402.1040818095207, 396.2246047258377, 390.3

In [19]:
# list out keys and values separately
key_list = list(word_to_ix.keys())
val_list = list(word_to_ix.values())

In [20]:
def similarity_cbow(word_1, word_2):
    
    # test word similarity
    print(word_1)
    print(word_2)
    w1_id = torch.tensor(word_to_ix[word_1], dtype=torch.long)
    w2_id = torch.tensor(word_to_ix[word_2], dtype=torch.long)
    w1_id = w1_id.to(device)
    w2_id = w2_id.to(device)
    
    word_1_vec = model.get_word_vector(w1_id)
    word_2_vec = model.get_word_vector(w2_id)
    
    # The norm of a vector (1D-matrix) is the square root of the sum of all the squared values within the vector.
    print(math.sqrt(torch.square(word_1_vec).sum()))    
    print(torch.linalg.norm(word_1_vec))
    print(torch.linalg.norm(word_2_vec))
    print(word_1_vec.dot(word_2_vec))
    
    word_distance = torch.linalg.norm(word_1_vec - word_2_vec)
    print("Distance between '{}' & '{}' : {:0.4f}".format(word_1, word_2, word_distance))
    word_similarity = (word_1_vec.dot(word_2_vec) / (torch.linalg.norm(word_1_vec) * torch.linalg.norm(word_2_vec)))
    print("Similarity between '{}' & '{}' : {:0.4f}".format(word_1, word_2, word_similarity))


In [21]:
similarity_cbow("neural", "network")

neural
network
2.60844889213183
tensor(2.6084, grad_fn=<LinalgVectorNormBackward0>)
tensor(3.1216, grad_fn=<LinalgVectorNormBackward0>)
tensor(-1.5512, grad_fn=<DotBackward0>)
Distance between 'neural' & 'network' : 4.4329
Similarity between 'neural' & 'network' : -0.1905


In [22]:
def predict_middle_word(prev_words, post_words):
    prev_words = prev_words.split()
    post_words = post_words.split()

    input_words= make_context_vector(prev_words + post_words, word_to_ix)
    input_words = input_words.to(device)
    output = model(input_words)
    out_ind = output.argmax(1)
#     print(word_to_ix)
#     out_word = word_to_ix.itos[out_ind.item()]
    out_word = key_list[val_list.index(out_ind.item())]
    print(prev_words, out_word, post_words)

In [23]:
predict_middle_word("a recurrent", "network is")
predict_middle_word("LSTM is", "recurrent neural")

['a', 'recurrent'] neural ['network', 'is']
['LSTM', 'is'] a ['recurrent', 'neural']


### Now that you saw how to create the CBOW model (word2vec), you should work on doing the "opposite" model, Skip-Gram ###

Skip-gram as you saw on the lectures, reverses the problem so you need to predict through the "fake task" the context of the input

In [24]:
# Generate Skip-gram pairs from the corpus
skipgram_data = []
for i in range(CONTEXT_SIZE, len(raw_text) - CONTEXT_SIZE):
    target_word = raw_text[i]
    context_words = []
    
    for j in range(-CONTEXT_SIZE, CONTEXT_SIZE + 1):
        if j != 0:
            context_word = raw_text[i + j]
            skipgram_data.append((target_word, context_word))

print(skipgram_data[:5])


[('Memory', 'Long'), ('Memory', 'Short-Term'), ('Memory', '(LSTM)'), ('Memory', 'is'), ('(LSTM)', 'Short-Term')]


In [25]:
import torch
import torch.nn as nn
import torch.optim as optim
import random

CONTEXT_SIZE = 2
embedding_dim = 10
epochs = 100

# Prepare vocabulary and mappings
word_to_ix = {word: i for i, word in enumerate(set(raw_text))}
ix_to_word = {i: word for word, i in word_to_ix.items()}
vocab_size = len(word_to_ix)

# Prepare skip-gram training data: (target, context) pairs
skipgram_data = []
for i in range(CONTEXT_SIZE, len(raw_text) - CONTEXT_SIZE):
    target_word = raw_text[i]
    target_idx = word_to_ix[target_word]
    for j in range(-CONTEXT_SIZE, CONTEXT_SIZE + 1):
        if j == 0:
            continue
        context_word = raw_text[i + j]
        context_idx = word_to_ix[context_word]
        skipgram_data.append((target_idx, context_idx))

# Define Skip-gram Model
class SkipGramModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.out = nn.Linear(embedding_dim, vocab_size)

    def forward(self, target_words):
        embeds = self.embeddings(target_words)        # [batch_size, embedding_dim]
        out = self.out(embeds)                         # [batch_size, vocab_size]
        return out

skip_gam_model = SkipGramModel(vocab_size, embedding_dim)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.SGD(skip_gam_model.parameters(), lr=0.01)

# Training loop
for epoch in range(epochs):
    total_loss = 0
    random.shuffle(skipgram_data)
    for target, context in skipgram_data:
        target_tensor = torch.tensor([target], dtype=torch.long)
        context_tensor = torch.tensor([context], dtype=torch.long)

        optimizer.zero_grad()
        output = skip_gam_model(target_tensor)
        loss = loss_fn(output, context_tensor)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}")

# Print embeddings for a few words
for word in list(word_to_ix.keys())[:10]:
    idx = word_to_ix[word]
    embed_vector = skip_gam_model.embeddings.weight[idx].detach().numpy()
    print(f"{word}: {embed_vector}")


Epoch 10/100, Loss: 2454.0110
Epoch 20/100, Loss: 2282.2374
Epoch 30/100, Loss: 2155.4649
Epoch 40/100, Loss: 2037.7013
Epoch 50/100, Loss: 1922.5938
Epoch 60/100, Loss: 1813.1953
Epoch 70/100, Loss: 1714.4663
Epoch 80/100, Loss: 1629.6002
Epoch 90/100, Loss: 1559.2354
Epoch 100/100, Loss: 1502.1798
tasks,: [ 1.3929735   0.8898699  -0.31551203 -2.2044137   1.6684431   2.6351197
  0.7272839  -0.38913408 -0.83881813  0.9689645 ]
make: [-2.0611656   1.1031476   0.8743024  -0.10556544  0.37860113  3.6695228
 -0.3262358  -0.19909309 -1.3070302  -0.62533826]
have: [ 0.6904707   0.5296067  -0.8630689  -0.9233398   1.2700373  -0.27783304
 -0.95466715 -1.1620792   2.01961     2.3705163 ]
to: [ 1.0710042  -0.3797032   1.0128124   1.2556654  -0.54505837 -2.4414716
  1.2120417  -1.914357    0.4289818   2.0584116 ]
tasks.: [-2.0520053   0.4827287   1.9571021   0.00705583  0.3150773  -0.26126122
  0.38849476  0.97035795  1.3915823   2.0594022 ]
model: [ 0.17343664  0.5374383   0.51133484  1.218472  

In [26]:
target_word

'small'

In [27]:
import torch
import torch.nn.functional as F

def predict_context_words(model, word_to_ix, ix_to_word, input_word, top_k=5):
    model.eval()
    if input_word not in word_to_ix:
        print(f"Word '{input_word}' not in vocabulary.")
        return
    
    input_idx = torch.tensor([word_to_ix[input_word]], dtype=torch.long)
    with torch.no_grad():
        output = model(input_idx)  # logits over vocab
        probs = F.softmax(output, dim=1)
        
    top_probs, top_indices = torch.topk(probs, top_k)
    predicted_words = [ix_to_word[idx.item()] for idx in top_indices[0]]
    predicted_probs = top_probs[0].tolist()

    print(f"Input word: '{input_word}'")
    print("Predicted context words:")
    for w, p in zip(predicted_words, predicted_probs):
        print(f"  {w} (prob: {p:.4f})")

predict_context_words(skip_gam_model, word_to_ix, ix_to_word, "feedforward", top_k=5)


Input word: 'feedforward'
Predicted context words:
  Unlike (prob: 0.2356)
  RNNs. (prob: 0.2107)
  networks, (prob: 0.1786)
  neural (prob: 0.1625)
  and (prob: 0.0472)


# Compare embeddings

In [ ]:
skipgram_embeddings = skip_gam_model.embeddings.weight.data
cbow_embeddings = model.embedding.weight.data

sample_word = "feedforward"
idx = word_to_ix[sample_word]

print(f"Skip-gram embedding for '{sample_word}':\n", skipgram_embeddings[idx])
print(f"CBOW embedding for '{sample_word}':\n", cbow_embeddings[idx])

Skip-gram embedding for 'feedforward':
 tensor([-1.3610, -3.4089,  0.1491, -0.6381, -1.1847, -0.0670, -2.7386,  0.9214,
         0.4940, -0.4432])
CBOW embedding for 'feedforward':
 tensor([ 0.2640, -0.8016, -0.9021,  0.8517, -0.6956,  0.1703, -0.8956,  0.7968,
        -1.2142, -1.8271])
